# 05 — Namespaces, using, and Aliasing

Namespaces solve the naming collision problem that C doesn't have a solution for. In C, if two libraries both define a function called `init()`, you're in trouble. In C++, each library can put its code inside its own namespace, and you call them as `LibA::init()` and `LibB::init()` — no conflict. This notebook covers namespaces, the `using` keyword, anonymous namespaces, `typedef`, and scope resolution.


## The Problem: Name Collisions

Imagine two libraries you want to use both define a function called `connect()`. In C, there is no mechanism to distinguish them — whoever gets linked second wins (or you get a linker error). This is the name collision problem.


In [ ]:
#include <iostream>

// Imagine these come from two different libraries:

// From "network_lib.h":
// void connect() { ... connects to a server ... }

// From "database_lib.h":
// void connect() { ... connects to a database ... }

// In C, you cannot have both. The linker will complain.
// Common C workarounds: prefix everything manually (net_connect, db_connect)
// C++ solution: namespaces.

std::cout << "Name collision is a real problem in large C projects." << std::endl;
std::cout << "Namespaces are C++'s solution." << std::endl;

## Defining a Namespace

Use the `namespace` keyword to group declarations. Outside the namespace, access members with the `::` scope resolution operator.


In [ ]:
#include <iostream>
#include <string>

namespace NetworkLib {
    std::string serverAddress = "192.168.1.1";

    void connect() {
        std::cout << "[NetworkLib] Connecting to " << serverAddress << std::endl;
    }

    void disconnect() {
        std::cout << "[NetworkLib] Disconnected." << std::endl;
    }
}

namespace DatabaseLib {
    std::string dbName = "myapp_db";

    void connect() {
        std::cout << "[DatabaseLib] Connecting to " << dbName << std::endl;
    }

    void disconnect() {
        std::cout << "[DatabaseLib] Disconnected from DB." << std::endl;
    }
}

// No collision — both connect() functions coexist
NetworkLib::connect();
DatabaseLib::connect();
NetworkLib::disconnect();
DatabaseLib::disconnect();
// Expected output:
// [NetworkLib] Connecting to 192.168.1.1
// [DatabaseLib] Connecting to myapp_db
// [NetworkLib] Disconnected.
// [DatabaseLib] Disconnected from DB.

## Nested Namespaces

Namespaces can be nested to create a hierarchy. Access nested members with chained `::` operators.


In [ ]:
#include <iostream>

namespace Company {
    namespace Graphics {
        int MAX_COLORS = 256;

        void render() {
            std::cout << "Rendering with " << MAX_COLORS << " colors" << std::endl;
        }
    }

    namespace Audio {
        int SAMPLE_RATE = 44100;

        void play() {
            std::cout << "Playing at " << SAMPLE_RATE << " Hz" << std::endl;
        }
    }
}

// Access nested namespace members
Company::Graphics::render();
Company::Audio::play();
std::cout << "Max colors: " << Company::Graphics::MAX_COLORS << std::endl;
// Expected output:
// Rendering with 256 colors
// Playing at 44100 Hz
// Max colors: 256

## The std Namespace

Everything in the C++ standard library lives in the `std` namespace. That is why you write `std::cout`, `std::string`, `std::vector`, etc. The `std::` prefix tells the compiler to look inside the standard library namespace.

This design means you can define your own `string` class or `cout` variable without conflicting with the standard library ones.


In [ ]:
#include <iostream>
#include <string>
#include <vector>

// Everything from the standard library is in std::
std::string message = "Hello from std::string";
std::vector<int> numbers;  // we'll cover vectors later
numbers.push_back(1);
numbers.push_back(2);
numbers.push_back(3);

std::cout << message << std::endl;
std::cout << "Vector size: " << numbers.size() << std::endl;

// You could define your own 'string' without conflict:
// class string { ... };  // this would be ::string, not std::string
// Expected output:
// Hello from std::string
// Vector size: 3

## using namespace

`using namespace std;` brings all names from `std` into the current scope so you can write `cout` instead of `std::cout`. It's convenient for short programs, but **dangerous in header files** — it pollutes the global namespace for every file that includes that header.

**Rule:** Never put `using namespace` in a header file. In source files (.cpp), use with caution.


In [ ]:
#include <iostream>
#include <string>

// Without using namespace:
std::cout << "Without using namespace: " << std::endl;
std::string s1 = "hello";
std::cout << s1 << std::endl;

In [ ]:
#include <iostream>
#include <string>

using namespace std;

// Now we can omit std::
cout << "With using namespace std: " << endl;
string s2 = "world";
cout << s2 << endl;

// But this brings EVERYTHING from std into scope.
// If you define a function called 'count', it clashes with std::count.
// In a header, this affects ALL files that include the header.

**Exercise 1:** Define a namespace called `MathUtils` containing a function `double square(double x)`. Call it from outside the namespace using:
1. The fully qualified name `MathUtils::square(...)`
2. A `using` declaration to bring just `square` into scope


In [ ]:
#include <iostream>

// Your code here

## using Declaration

`using std::cout;` brings **only** `cout` from `std` into scope. This is much safer than `using namespace std;` because it introduces only the specific name you need, reducing the risk of conflicts.


In [ ]:
#include <iostream>
#include <string>

// Bring in only what we need
using std::cout;
using std::endl;
using std::string;

// Now these work without std::
string greeting = "Hello from using declaration";
cout << greeting << endl;

// But std::cin still requires std:: — we didn't bring it in
// std::cin >> ...   <-- this still needs the prefix

// Inside a function, using declarations are scoped to the function:
// void foo() {
//     using std::string;  // only visible inside foo()
// }
// Expected output:
// Hello from using declaration

## Anonymous Namespaces

An unnamed `namespace { }` gives its contents **internal linkage** — they are only visible within the current translation unit (file). This is the C++ replacement for C's `static` at file scope.


In [ ]:
#include <iostream>

// In C, you'd write: static int fileLocalCounter = 0;
// In C++, prefer anonymous namespace:
namespace {
    int fileLocalCounter = 0;

    void incrementCounter() {
        fileLocalCounter++;
    }
}

// These are usable within this file, but not visible to other translation units.
incrementCounter();
incrementCounter();
incrementCounter();
std::cout << "Counter: " << fileLocalCounter << std::endl;
// Expected output:
// Counter: 3

## typedef

In C++98, `typedef` creates type aliases. This is useful for shortening long type names, creating semantic type names, and defining function pointer types.


In [ ]:
#include <iostream>
#include <string>

// Simple type aliases
typedef unsigned int uint;
typedef unsigned char byte;
typedef long long int64;

uint score = 100;
byte flags = 0xFF;
int64 bigValue = 9000000000LL;

std::cout << "score: " << score << std::endl;
std::cout << "flags: " << (int)flags << std::endl;
std::cout << "bigValue: " << bigValue << std::endl;
// Expected output:
// score: 100
// flags: 255
// bigValue: 9000000000

In [ ]:
#include <iostream>

// Function pointer typedef — the C syntax is cryptic without typedef
// Without typedef: void (*Handler)(int, const char*);
// With typedef:
typedef void (*Handler)(int, const char *);

void onError(int code, const char *msg) {
    std::cout << "Error " << code << ": " << msg << std::endl;
}

void onWarning(int code, const char *msg) {
    std::cout << "Warning " << code << ": " << msg << std::endl;
}

// Now Handler is a readable type name for this function pointer
Handler h = onError;
h(404, "Not found");

h = onWarning;
h(301, "Moved permanently");
// Expected output:
// Error 404: Not found
// Warning 301: Moved permanently

**Exercise 2:** Create a `typedef` for a function pointer that takes two `int`s and returns an `int`. Write two functions (`add` and `multiply`) matching that signature, and call them via the typedef.


In [ ]:
#include <iostream>

// Your code here

## Scope Resolution ::

The `::` operator serves multiple purposes:
1. Accessing namespace members: `std::cout`
2. Accessing class static members: `MyClass::count` (covered later)
3. Accessing global scope when a local name shadows it


In [ ]:
#include <iostream>

// Global variable
int value = 100;

void showShadowing() {
    // Local variable shadows the global
    int value = 42;

    std::cout << "Local value: " << value << std::endl;    // local: 42
    std::cout << "Global value: " << ::value << std::endl; // global: 100
}

showShadowing();
// Expected output:
// Local value: 42
// Global value: 100

In [ ]:
#include <iostream>

// Demonstrating multiple uses of ::

namespace Config {
    int MAX_CONNECTIONS = 10;

    void print() {
        std::cout << "Max connections: " << MAX_CONNECTIONS << std::endl;
    }
}

// 1. Namespace access
Config::print();
Config::MAX_CONNECTIONS = 20;

// 2. After modification
std::cout << "Updated: " << Config::MAX_CONNECTIONS << std::endl;
// Expected output:
// Max connections: 10
// Updated: 20

## Final Exercise

Create two namespaces `Geometry` and `Physics`, each with a function called `calculate()` that does something different (e.g., Geometry calculates area of a circle given a radius, Physics calculates kinetic energy given mass and velocity). Call both from outside without using `using namespace`. Also create a `typedef` for a pair of doubles called `Point2D` (use a struct for the pair).


In [ ]:
#include <iostream>

// Your code here

## Modern C++ (C++11 and Beyond)

C++11 introduced a cleaner alias syntax with `using`, and later standards added more namespace conveniences.


In [ ]:
#include <iostream>
#include <vector>

// C++11: 'using' alias syntax — cleaner than typedef, works with templates
using Uint = unsigned int;
using IntVector = std::vector<int>;

// With typedef, template aliases were impossible. With 'using', they work:
// template<typename T>
// using Vec = std::vector<T>;   // C++11 alias template — not possible with typedef

Uint count = 5;
IntVector nums = {1, 2, 3, 4, 5};
std::cout << "count: " << count << std::endl;
std::cout << "nums size: " << nums.size() << std::endl;

In [ ]:
#include <iostream>

// C++11: inline namespaces — members are visible in the enclosing namespace
// Useful for library versioning
namespace MyLib {
    inline namespace v2 {
        void greet() {
            std::cout << "Hello from MyLib v2!" << std::endl;
        }
    }
    namespace v1 {
        void greet() {
            std::cout << "Hello from MyLib v1!" << std::endl;
        }
    }
}

MyLib::greet();     // calls v2 (inline) — the default
MyLib::v1::greet(); // explicit v1
MyLib::v2::greet(); // explicit v2

// C++17: nested namespace shorthand
// namespace A::B::C { ... }   instead of namespace A { namespace B { namespace C { } } }
// Expected output:
// Hello from MyLib v2!
// Hello from MyLib v1!
// Hello from MyLib v2!